In [1]:
import pandas as pd
import csv
import subprocess
import time
import re
import numpy as np
import json
import requests
from tqdm import tqdm
from pydantic import BaseModel, ValidationError
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df = pd.read_csv("/content/drive/MyDrive/term_paper_masha/bank_reviews_dataset.csv", sep=";", encoding="utf-8-sig", quoting=csv.QUOTE_ALL, escapechar="\\")

df.head()

,id,link,date,title,text,rating,city,source
0,12899382,https://www.banki.ru/services/responses/bank/r...,2026-02-08,"Блокировка карты вместе с деньгами, бросание т...",Сегодня при снятии (попытке снять свои деньги)...,1.0,Рязань,banki
1,12898481,https://www.banki.ru/services/responses/bank/r...,2026-02-08,Самый ужасный банк в обслуживании,Самый ужасный/ не гибкий банк Росии и крупнейш...,2.0,Москва,banki
2,12898275,https://www.banki.ru/services/responses/bank/r...,2026-02-08,"Граждане, бегите из этого банка, если хотите, ...","В Газпромбанке решила открыть вклад , но так к...",1.0,Новочеркасск,banki
3,12898259,https://www.banki.ru/services/responses/bank/r...,2026-02-08,Как Газпромбанк получает млрд на деньгах гражд...,"09.01.26 ГПБ заблокировал карты, я сразу предо...",1.0,Новочеркасск,banki
4,12897965,https://www.banki.ru/services/responses/bank/r...,2026-02-07,Блокировка карты,"Заказал, дебетовую карту Газпромбанк а на сайт...",1.0,Барнаул,banki


In [8]:
!pip install pandas tqdm pydantic

In [9]:
!apt-get update -y
!apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,955 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,646 kB]
Get:13 http://security.ubuntu.com/ubunt

In [22]:
!pkill ollama || true

In [23]:
!nohup ollama serve > ollama.log 2>&1 &

In [15]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [24]:
!pip install requests

In [25]:
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(10)
print("Ollama server started")

Ollama server started


In [26]:
!curl http://127.0.0.1:11434

Ollama is running

In [28]:
!ollama pull qwen3:1.7b

In [30]:
!curl http://localhost:11434/api/tags

{"models":[{"name":"qwen3:1.7b","model":"qwen3:1.7b","modified_at":"2026-05-25T18:50:03.428474905Z","size":1359293444,"digest":"8f68893c685c3ddff2aa3fffce2aa60a30bb2da65ca488b61fff134a4d1730e7","details":{"parent_model":"","format":"gguf","family":"qwen3","families":["qwen3"],"parameter_size":"2.0B","quantization_level":"Q4_K_M"}}]}

In [29]:
!ollama list

NAME          ID              SIZE      MODIFIED               
qwen3:1.7b    8f68893c685c    1.4 GB    Less than a second ago    


проверка на респонс

In [31]:
ALLOWED_LABELS = {
    "CARDS_POS", "CARDS_NEG",
    "DIGITAL_POS", "DIGITAL_NEG",
    "SUPPORT_POS", "SUPPORT_NEG",
    "PAYMENTS_POS", "PAYMENTS_NEG",
    "CREDITS_POS", "CREDITS_NEG",
    "FEES_POS", "FEES_NEG",
    "OFFICE_POS", "OFFICE_NEG",
    "ATM_POS", "ATM_NEG",
    "FRAUD_POS", "FRAUD_NEG",
    "INSURANCE_POS", "INSURANCE_NEG",
    "INVESTMENTS_POS", "INVESTMENTS_NEG",
    "PREMIUM_POS", "PREMIUM_NEG",
    "OTHER_POS", "OTHER_NEG"
}


class AspectItem(BaseModel):
    label: str
    evidence: str


class ABSAResult(BaseModel):
    aspects: list[AspectItem]

In [32]:
def build_prompt(text: str) -> str:
    return f"""
/no_think
Ты размечаешь отзывы клиентов банка для задачи Aspect-Based Sentiment Analysis.

Нужно найти в отзыве фрагменты текста, которые относятся к банковским аспектам,
и определить тональность каждого аспекта. Не рассуждай. Не объясняй.

Разрешенные labels:

CARDS_POS / CARDS_NEG — карты, выпуск, доставка, блокировка, пин-код, кэшбек, бонусы по карте.
DIGITAL_POS / DIGITAL_NEG — мобильное приложение, интернет-банк, личный кабинет, вход, авторизация.
SUPPORT_POS / SUPPORT_NEG — поддержка, чат, горячая линия, оператор, контактный центр.
PAYMENTS_POS / PAYMENTS_NEG — платежи, переводы, СБП, оплата, зачисление, списание.
CREDITS_POS / CREDITS_NEG — кредиты, ипотека, автокредит, график платежей, досрочное погашение.
FEES_POS / FEES_NEG — комиссии, платное обслуживание, скрытые списания, тарифы.
OFFICE_POS / OFFICE_NEG — офис, отделение, касса, сотрудники в офисе, очередь.
ATM_POS / ATM_NEG — банкоматы, снятие наличных, внесение наличных.
FRAUD_POS / FRAUD_NEG — мошенничество, подозрительные операции, безопасность, 115-ФЗ, блокировки из-за финмониторинга.
INSURANCE_POS / INSURANCE_NEG — страховки, страхование, финансовая защита.
INVESTMENTS_POS / INVESTMENTS_NEG — инвестиции, брокерский счет, ИИС, ПИФ.
PREMIUM_POS / PREMIUM_NEG — премиальное обслуживание, персональный менеджер, private banking.
OTHER_POS / OTHER_NEG — важный банковский аспект, который не попал в категории выше.

Правила:
1. Верни только JSON, без пояснений. Не рассуждай. Не объясняй.
2. Не придумывай аспекты, которых нет в тексте.
3. evidence должен быть точной цитатой из отзыва.
4. Один отзыв может содержать несколько аспектов.
5. Если аспектов нет, верни {{"aspects": []}}.
6. Sentiment определяй по отношению клиента к конкретному аспекту.

Формат ответа:
{{
  "aspects": [
    {{
      "label": "CARDS_NEG",
      "evidence": "заблокировали карту"
    }}
  ]
}}

Отзыв:
{text}

Верни только JSON.



Твоя задача — найти ВСЕ аспекты в отзыве банка.

Один отзыв может содержать несколько аспектов.

Не останавливайся после первого найденного аспекта.

Разрешенные labels:
DIGITAL_NEG, DIGITAL_POS, SUPPORT_NEG, SUPPORT_POS, CARDS_NEG, CARDS_POS, ATM_NEG, ATM_POS, FRAUD_NEG, FRAUD_POS, OTHER_NEG, OTHER_POS.

Формат:
{{"aspects":[{{"label":"DIGITAL_NEG","evidence":"Мобильное приложение ужасное"}}]}}

Отзыв:
{text}
"""

In [33]:
def parse_llm_json(raw_text: str) -> dict:
    raw_text = raw_text.strip()

    if raw_text.startswith("```json"):
        raw_text = raw_text.replace("```json", "").replace("```", "").strip()
    elif raw_text.startswith("```"):
        raw_text = raw_text.replace("```", "").strip()

    return json.loads(raw_text)


def validate_result(result: dict) -> dict:
    parsed = ABSAResult(**result)

    clean_aspects = []

    for item in parsed.aspects:
        if item.label in ALLOWED_LABELS and item.evidence.strip():
            clean_aspects.append({
                "label": item.label,
                "evidence": item.evidence.strip()
            })

    return {"aspects": clean_aspects}


def annotate_review_local(text, model="qwen3:1.7b"):

    prompt = build_prompt(text)

    response = requests.post(
      "http://localhost:11434/api/generate",
      json={
          "model": model,
          "prompt": prompt,
          "stream": False,
          "format": "json",
          "options": {
              "temperature": 0,
              "num_predict": 256,
              "num_ctx": 2048,
              "think": False
          }
      },
      timeout=300
  )

    raw = response.json()["response"]

    try:
        result = json.loads(raw)
        return result

    except Exception as e:
        return {
            "aspects": [],
            "error": str(e),
            "raw_response": raw
        }

In [34]:
def split_review_into_chunks(text, max_chars=1200):
    text = str(text).strip()

    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) <= max_chars:
            current += " " + sentence
        else:
            if current.strip():
                chunks.append(current.strip())
            current = sentence

    if current.strip():
        chunks.append(current.strip())

    return chunks

In [35]:
def extract_json_from_text(raw: str):
    raw = raw.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    match = re.search(r'\{.*\}', raw, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in response: {raw[:300]}")

    return json.loads(match.group(0))


def annotate_chunk_local(text: str, model: str = "qwen3:1.7b") -> dict:
    prompt = build_prompt(text)

    try:
        response = requests.post(
          "http://localhost:11434/api/generate",
          json={
              "model": model,
              "prompt": prompt,
              "stream": False,
              "format": "json",
              "options": {
                  "temperature": 0,
                  "num_predict": 256,
                  "num_ctx": 2048,
                  "think": False
              }
          },
          timeout=300
      )

        data = response.json()
        raw = data.get("response", "")

        if not raw.strip():
            return {
                "aspects": [],
                "error": "empty_response",
                "ollama_response": data
            }

        result = extract_json_from_text(raw)

        for aspect in result.get("aspects", []):
            if "evaluation" in aspect and "evidence" not in aspect:
                aspect["evidence"] = aspect.pop("evaluation")

        result = validate_result(result)

        return result

    except Exception as e:
        return {
            "aspects": [],
            "error": str(e),
            "raw_response": raw if "raw" in locals() else "",
            "ollama_response": data if "data" in locals() else None
        }

In [36]:
test_text = """
Мобильное приложение ужасное, постоянно вылетает.
В поддержке сидят некомпетентные специалисты, не решили мою проблему
"""

print(annotate_chunk_local(test_text))

{'aspects': [{'label': 'DIGITAL_NEG', 'evidence': 'Мобильное приложение ужасное'}, {'label': 'SUPPORT_NEG', 'evidence': 'В поддержке сидят некомпетентные специалисты'}]}


In [37]:
def annotate_long_review_local(text: str, model: str = "qwen3:1.7b") -> dict:
    chunks = split_review_into_chunks(text, max_chars=700)

    all_aspects = []
    errors = []

    for chunk_id, chunk in enumerate(chunks):
        result = annotate_chunk_local(chunk, model=model)

        if "error" in result:
            errors.append({
                "chunk_id": chunk_id,
                "error": result["error"],
                "raw_response": result.get("raw_response", "")
            })

        for aspect in result.get("aspects", []):
            aspect["chunk_id"] = chunk_id
            all_aspects.append(aspect)

    seen = set()
    unique_aspects = []

    for aspect in all_aspects:
        key = (
            aspect.get("label", ""),
            aspect.get("evidence", "").lower().strip()
        )

        if key not in seen:
            seen.add(key)
            unique_aspects.append(aspect)

    output = {
        "aspects": unique_aspects
    }

    if errors:
        output["errors"] = errors

    return output

In [38]:
full_df = df.copy()

predictions = []
for _, row in tqdm(full_df.iterrows(), total=len(full_df)):
    text = str(row["text"])
    result = annotate_long_review_local(text)
    predictions.append({"id": row["id"], "text": text, "prediction": result})

    time.sleep(0.3)

100%|██████████| 10/10 [01:00<00:00,  6.01s/it]


In [39]:
with open("absa_predictions_raw.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print("Saved:", len(predictions))

In [40]:
def find_span(text: str, evidence: str):
    start = text.lower().find(evidence.lower())

    if start == -1:
        return None

    end = start + len(evidence)
    return start, end


def convert_to_label_studio(df_part: pd.DataFrame, predictions: list) -> list:
    pred_by_id = {str(item["id"]): item["prediction"] for item in predictions}

    tasks = []

    for _, row in df_part.iterrows():
        review_id = str(row["id"])
        text = str(row["text"])

        result_items = []

        pred = pred_by_id.get(review_id, {"aspects": []})

        for aspect in pred.get("aspects", []):
            label = aspect.get("label")
            evidence = aspect.get("evidence")

            if not label or not evidence:
                continue

            span = find_span(text, evidence)

            if span is None:
                continue

            start, end = span

            result_items.append({
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": start,
                    "end": end,
                    "text": text[start:end],
                    "labels": [label]
                }
            })

        task = {
            "data": {
                "id": review_id,
                "text": text,
                "title": str(row.get("title", "")),
                "rating": str(row.get("rating", "")),
                "source": str(row.get("source", ""))
            },
            "predictions": [
                {
                    "model_version": "llm_absa_v1",
                    "result": result_items
                }
            ]
        }

        tasks.append(task)

    return tasks

In [41]:
label_studio_tasks = convert_to_label_studio(sample_df, predictions)

with open("label_studio_preannotations.json", "w", encoding="utf-8") as f:
    json.dump(label_studio_tasks, f, ensure_ascii=False, indent=2)

#Results


In [50]:
RAW_PREDICTIONS_PATH = "absa_predictions_raw.json"
with open(RAW_PREDICTIONS_PATH, "r", encoding="utf-8") as f:
    raw_predictions = json.load(f)

teacher_path = "/content/drive/MyDrive/term_paper_masha/predictions_clean_full_aspects.csv"
manual_path = "/content/drive/MyDrive/term_paper_masha/labeled_reviews_with_additional_with_labels_list.csv"

In [51]:
def extract_labels_with_repeats(item):
    aspects = item.get("prediction", {}).get("aspects", [])
    labels = []
    for aspect in aspects:
        label = aspect.get("label")
        if label is not None:
            labels.append(label)

    return ", ".join(labels)

In [52]:
qwen_df = pd.DataFrame([{"id": item.get("id"), "text": item.get("text"), "prediction": json.dumps(item.get("prediction", {}), ensure_ascii=False),
        "labels_list": extract_labels_with_repeats(item)} for item in raw_predictions])
qwen_df["id"] = qwen_df["id"].astype(str)
qwen_df.head()

,id,text,prediction,labels_list
0,12899382,Сегодня при снятии (попытке снять свои деньги)...,"{""aspects"": [{""label"": ""FRAUD_NEG"", ""evidence""...","FRAUD_NEG, CARDS_NEG, ATM_NEG, OTHER_NEG, FRAU..."
1,12898481,Самый ужасный/ не гибкий банк Росии и крупнейш...,"{""aspects"": [{""label"": ""DIGITAL_NEG"", ""evidenc...","DIGITAL_NEG, SUPPORT_NEG, CARDS_NEG, ATM_NEG, ..."
2,12898275,"В Газпромбанке решила открыть вклад , но так к...","{""aspects"": [{""label"": ""DIGITAL_NEG"", ""evidenc...","DIGITAL_NEG, CARDS_NEG, PAYMENTS_NEG, FRAUD_NE..."
3,12898259,"09.01.26 ГПБ заблокировал карты, я сразу предо...","{""aspects"": [{""label"": ""CARDS_NEG"", ""evidence""...","CARDS_NEG, DIGITAL_NEG, SUPPORT_NEG, FRAUD_NEG..."
4,12897965,"Заказал, дебетовую карту Газпромбанк а на сайт...","{""aspects"": [{""label"": ""DIGITAL_NEG"", ""evidenc...","DIGITAL_NEG, SUPPORT_NEG, CARDS_NEG, ATM_NEG, ..."


In [77]:
output_predictions_path = "qwen_zero_shot_predictions_with_labels_list.csv"
qwen_df.to_csv(output_predictions_path, sep=";", index=False, encoding="utf-8-sig")

print("Saved to:", output_predictions_path)

Saved to: qwen_zero_shot_predictions_with_labels_list.csv


In [67]:
ALL_LABELS = [
    "CARDS_POS", "CARDS_NEG",
    "DIGITAL_POS", "DIGITAL_NEG",
    "SUPPORT_POS", "SUPPORT_NEG",
    "PAYMENTS_POS", "PAYMENTS_NEG",
    "CREDITS_POS", "CREDITS_NEG",
    "FEES_POS", "FEES_NEG",
    "OFFICE_POS", "OFFICE_NEG",
    "ATM_POS", "ATM_NEG",
    "FRAUD_POS", "FRAUD_NEG",
    "INSURANCE_POS", "INSURANCE_NEG",
    "INVESTMENTS_POS", "INVESTMENTS_NEG",
    "PREMIUM_POS", "PREMIUM_NEG",
    "OTHER_POS", "OTHER_NEG"
]

In [68]:
def parse_labels(x):
    if pd.isna(x):
        return []
    x = str(x)
    labels = re.findall(r"[A-Z]+_(?:POS|NEG)", x)
    labels = [label for label in labels if label in ALL_LABELS]

    return sorted(set(labels))

In [69]:
def read_csv(path):
    try:
      return pd.read_csv(path, sep=";", encoding="utf-8-sig")
    except:
        return pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig", on_bad_lines="skip")

In [70]:
teacher_df = read_csv(teacher_path)
manual_df = read_csv(manual_path)

print("qwen_df:", qwen_df.shape)
print("teacher_df:", teacher_df.shape)
print("manual_df:", manual_df.shape)

print("qwen columns:", qwen_df.columns.tolist())
print("teacher columns:", teacher_df.columns.tolist())
print("manual columns:", manual_df.columns.tolist())

qwen_df: (10, 5)
teacher_df: (6398, 5)
manual_df: (311, 15)
qwen columns: ['id', 'text', 'prediction', 'labels_list', 'pred_labels']
teacher columns: ['id', 'title', 'text', 'prediction', 'labels_list']
manual columns: ['annotation_id', 'annotator', 'city', 'created_at', 'date', 'id', 'label', 'lead_time', 'link', 'rating', 'source', 'text', 'title', 'updated_at', 'labels_list']


In [71]:
qwen_df["pred_labels"] = qwen_df["labels_list"].apply(parse_labels)
teacher_df["id"] = teacher_df["id"].astype(str)
manual_df["id"] = manual_df["id"].astype(str)
teacher_df["true_labels"] = teacher_df["labels_list"].apply(parse_labels)
manual_df["true_labels"] = manual_df["labels_list"].apply(parse_labels)

In [72]:
def soft_accuracy_no_extra_penalty(true_labels, pred_labels):
    true_set = set(true_labels)
    pred_set = set(pred_labels)
    if len(true_set) == 0:
        return np.nan

    return len(true_set.intersection(pred_set)) / len(true_set)

In [73]:
def calculate_one_dataset_metrics(base_df, pred_df):
    eval_df = base_df[["id", "true_labels"]].merge(pred_df[["id", "pred_labels"]], on="id", how="inner")
    eval_df["soft_accuracy"] = eval_df.apply(lambda row: soft_accuracy_no_extra_penalty(row["true_labels"],
            row["pred_labels"]), axis=1)

    mlb = MultiLabelBinarizer(classes=ALL_LABELS)
    y_true = mlb.fit_transform(eval_df["true_labels"])
    y_pred = mlb.transform(eval_df["pred_labels"])
    metrics = {
        "Precision": precision_score(y_true, y_pred, average="micro", zero_division=0) * 100,
        "Recall": recall_score(y_true, y_pred, average="micro", zero_division=0) * 100,
        "F1": f1_score(y_true, y_pred, average="micro", zero_division=0) * 100,
        "Exact Match Accuracy": accuracy_score(y_true, y_pred) * 100,
        "Soft Accuracy": eval_df["soft_accuracy"].mean() * 100}

    return metrics, eval_df

In [74]:
teacher_metrics, teacher_eval_df = calculate_one_dataset_metrics(base_df=teacher_df, pred_df=qwen_df)
manual_metrics, manual_eval_df = calculate_one_dataset_metrics(base_df=manual_df, pred_df=qwen_df)

In [75]:
metrics_table = pd.DataFrame({"Qwen-generated labels": teacher_metrics, "Manual labels": manual_metrics}).round(2)

metrics_table

,Qwen-generated labels,Manual labels
Precision,40.82,57.14
Recall,57.14,71.79
F1,47.62,63.64
Exact Match Accuracy,0.00,0.00
Soft Accuracy,56.83,77.74


In [78]:
output_metrics_path = "qwen_zero_shot_metrics.xlsx"

with pd.ExcelWriter(output_metrics_path, engine="openpyxl") as writer:
    metrics_table.to_excel(writer, sheet_name="metrics")
    teacher_eval_df.to_excel(writer, sheet_name="qwen_teacher_comparison", index=False)
    manual_eval_df.to_excel(writer, sheet_name="qwen_manual_comparison", index=False)

print("Saved to:", output_metrics_path)

Saved to: qwen_zero_shot_metrics.xlsx
